[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [APIs and JSON](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)

# Hosting an API


## What you will be able to do

Turn a FastAPI app into files a host can run: `main.py`, a `requirements.txt` pinned to the versions
you tested, a start command that listens on the port the host names, and a secret read from the
environment. Run the app with that command and check it before it leaves, then put it on a public
address with Render's free tier, and run the same checks there.


## The idea

### The problem

Every app in this guide ran inside a notebook, at `127.0.0.1`, which only the notebook's own machine
can reach, and only while the notebook runs. For another program to call it, an API needs a machine
that keeps running, a public address, and HTTPS.

It also needs everything the notebook gave it without being asked: the packages it imports, a
command that starts it, and its secrets. A secret cannot be typed into a cell on a machine nobody
sits at, and it must not be written into files that go into a repository, where anyone who can read
the repository can read the secret.

### What hosting an API is

> **Hosting** an API is running it on a server that other programs can reach at a public address. A
> **host** such as Render builds the app from its files with a **build command**, which installs its
> `requirements.txt`, and runs it with a **start command**, such as
> `uvicorn main:app --host 0.0.0.0 --port $PORT`. The host chooses the port and passes it in the
> `PORT` environment variable, and the app listens on `0.0.0.0`, every network address of its
> machine, so that the host's traffic reaches it. The host gives the app an address with HTTPS, and
> restarts it when it stops. **Environment variables** also carry the app's **secrets**, which are
> set in the host's dashboard, so that no file holds them.

### Why it works that way

- **The files are all the host knows.** A host starts from a repository, not from your notebook, so
  everything the app needs has to be in its files, and a package missing from `requirements.txt` is
  missing on the host.
- **Pinned versions install what you tested.** `fastapi==0.141.1` installs the version the notebook
  ran, where a bare `fastapi` installs whatever is newest on the day the host builds.
- **The host chooses the port.** Render sets `PORT`, which defaults to 10000, and an app listening on
  a port of its own choosing is not where Render sends requests. `$PORT` in the start command reads
  the variable.
- **`127.0.0.1` answers only its own machine.** A host's traffic reaches the app from outside, so
  the app has to listen on `0.0.0.0`, as Render's documentation requires.
- **Configuration lives in the environment.** The Twelve-Factor App states the rule as "Store config
  in the environment", and tests it by asking whether the code could be made open source at any
  moment without compromising any credentials.
- **A free service sleeps.** Render spins down a free web service after 15 minutes without traffic,
  and the next request waits about a minute while it starts again, so a client needs a long timeout
  for its first call, and the retries of the **Errors and Retries** notebook.

### Where this shows up

Render's own guide to deploying a FastAPI app uses this build command and this start command.
FastAPI's documentation lists what a deployment has to settle: HTTPS, running on startup, restarts,
replication, memory, and the steps before starting. Hugging Face Spaces, which now needs a paid plan
to host an app like this one, tells its users not to hard-code secrets, and to add them in a Space's
settings instead. Colab keeps a secret in its Secrets panel, as the **Authentication** notebook
showed, and this notebook's last check reads the hosted app's key from there.

### What this notebook covers

- An app in a file of its own, `main.py`, with its data, its routes and a key it reads at startup
- `requirements.txt`, pinned to the versions this notebook runs
- The host's start command, run on this machine, with the port taken from `PORT`
- A route that refuses a request without the key, and a key that is in no file
- Checks that send a client's first requests, written once for any address
- The files packed for upload, and the steps that deploy them with GitHub and Render
- The same checks, run against the app on this machine and at its hosted address
- Five errors, from a start command that names the wrong file to a key written into a file

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import subprocess
import tempfile
import time
from pathlib import Path

import requests

folder = Path(tempfile.mkdtemp())
(folder / "main.py").write_text('''from fastapi import FastAPI

app = FastAPI()


@app.get("/")
def home():
    return {"stations": 4}
''')

server = subprocess.Popen(["uvicorn", "main:app", "--port", "8040"], cwd=folder,
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)                                       # long enough for uvicorn to start
print(requests.get("http://127.0.0.1:8040/", timeout=10).json())
server.terminate()
```

```
{'stations': 4}
```

A file, a command, and an app running as a process of its own, which is how a host runs one. The rest
is what a host adds: its own port, the app's secrets, and a public address.


## Setup

Thirteen imports, the last of them the practice API.

- `requests` sends requests to the app, on this machine and at its hosted address
- `subprocess` runs the host's start command as a process of its own
- `os` gives that process its environment variables
- `secrets` makes an API key, with `token_urlsafe`
- `time` waits for the app to start
- `version`, from `importlib.metadata`, reads a package's installed version, for `requirements.txt`
- `zipfile` packs the app's files for upload
- `shutil` removes the scratch folder at the end
- `urllib.request` fetches the practice API's code in Colab
- `Path` builds the paths of the app's files, and checks whether the practice API's code is here
- `sys` tells this cell, and the cells that download a file or read a secret, whether the notebook
  is running in Colab
- `importlib` reloads the practice API, so running this cell again uses its current code
- `practice_api` is the server this guide talks to, whose stations the app's are checked against

The app's files go in `scratch/station-api`, which this cell makes and the notebook's last cell
removes. No cell deploys anything: the deploy happens on GitHub's and Render's websites, in steps
this notebook lists.


In [1]:
import importlib
import os
import secrets
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from importlib.metadata import version
from pathlib import Path

import requests

PRACTICE_API = "https://raw.githubusercontent.com/johnfisher-ai/Python-Visual-Guides/main/notebooks/apis-and-json/practice_api.py"

if "google.colab" in sys.modules or not Path("practice_api.py").exists():
    urllib.request.urlretrieve(PRACTICE_API, "practice_api.py")    # in Colab, on every run

import practice_api
importlib.reload(practice_api)    # runs the file as it is now, not a copy imported earlier

BASE = practice_api.start()
PROJECT = Path("scratch/station-api")
PROJECT.mkdir(parents=True, exist_ok=True)
print("The practice API is running at", BASE, "| the app's files go in", PROJECT)


The practice API is running at http://127.0.0.1:8765 | the app's files go in scratch/station-api


## Worked examples

### An app in a file of its own

A host runs files, not cells. The app goes into `main.py`, in the folder that becomes its
repository. It carries its own copy of the stations, since the practice API will not be on the host,
and it reads its API key from the environment when it starts. Creating a plan needs that key.
`%%writefile`, as in the **A Real Client** notebook, saves the cell as the file:


In [2]:
%%writefile scratch/station-api/main.py
"""The weather stations API, as a host runs it: uvicorn main:app --host 0.0.0.0 --port $PORT"""

import itertools
import os
import secrets
from typing import Annotated

from fastapi import Depends, FastAPI, Header, HTTPException, Response, status
from pydantic import BaseModel, ConfigDict, Field

API_KEY = os.environ["STATION_API_KEY"]          # set on the host, and never written in a file

STATIONS = {
    "bergen": {"id": "bergen", "name": "Bergen", "latitude": 60.39, "longitude": 5.32},
    "oslo": {"id": "oslo", "name": "Oslo", "latitude": 59.91, "longitude": 10.75},
    "svalbard": {"id": "svalbard", "name": "Svalbard", "latitude": 78.22, "longitude": 15.65},
    "tromso": {"id": "tromso", "name": "Tromso", "latitude": 69.65, "longitude": 18.96},
}

app = FastAPI(title="Weather stations")
plans = {}
plan_ids = itertools.count(1)


class Plan(BaseModel):
    model_config = ConfigDict(extra="forbid")

    name: str = Field(min_length=1, max_length=50)
    latitude: float = Field(ge=-90, le=90, strict=True)
    longitude: float = Field(ge=-180, le=180, strict=True)


def require_key(x_api_key: Annotated[str | None, Header()] = None):
    """Refuse a request that does not carry the API key."""
    if x_api_key is None or not secrets.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail="send a valid API key in X-API-Key")


@app.get("/health")
def health():
    return {"status": "ok"}


@app.get("/stations")
def list_stations():
    return [{"id": found["id"], "name": found["name"]} for found in STATIONS.values()]


@app.get("/stations/{station_id}")
def read_station(station_id: str):
    if station_id not in STATIONS:
        raise HTTPException(status_code=404, detail=f"no station with id {station_id!r}")
    return STATIONS[station_id]


@app.post("/plans", status_code=status.HTTP_201_CREATED, dependencies=[Depends(require_key)])
def create_plan(plan: Plan, response: Response):
    created = {"id": next(plan_ids), **plan.model_dump()}
    plans[created["id"]] = created
    response.headers["Location"] = f"/plans/{created['id']}"
    return created


@app.get("/plans")
def list_plans():
    return list(plans.values())


Writing scratch/station-api/main.py


The app is a file now, and the key is not in it: `os.environ["STATION_API_KEY"]` reads the key from
the environment when the app starts. `require_key` is a dependency listed in the decorator's
`dependencies`, which FastAPI's documentation describes for a dependency whose check a route needs
and whose value it does not. `secrets.compare_digest` compares the key sent with the real one using
what Python's documentation calls a constant-time compare, to reduce the risk of timing attacks.

### Requirements, pinned to what ran

The host installs what `requirements.txt` lists, and nothing else. Pinning each package to the
version this notebook runs, read with `version`, has the host install what was tested:


In [3]:
requirements = "".join(f"{package}=={version(package)}\n" for package in ["fastapi", "uvicorn"])
(PROJECT / "requirements.txt").write_text(requirements)

print(requirements, end="")
print(sorted(path.name for path in PROJECT.iterdir()))


fastapi==0.141.1
uvicorn==0.52.4
['main.py', 'requirements.txt']


These are the versions Colab has, and the site's own `requirements.txt` pins the same. FastAPI brings
Starlette and Pydantic with it, at versions its own requirements allow. `requests` is not listed,
because the app does not import it: the notebook does.

### The host's start command, on this machine

Render runs `uvicorn main:app --host 0.0.0.0 --port $PORT` in the app's folder. `main:app` is the
object `app` in the file `main.py`, as the **Your First API Server** notebook said, and the shell
replaces `$PORT` with the `PORT` environment variable before the command runs. On your own
computer, `0.0.0.0` would also accept connections from other computers on your network, which a
firewall may ask about, so this notebook runs the command with `127.0.0.1` in its place, and
otherwise exactly as the host runs it. `start` runs a command in a folder with the environment it is
given, `wait_until_up` asks for `/health` until the app answers, and `stop` ends the process:


In [4]:
START = "uvicorn main:app --host 0.0.0.0 --port $PORT"
ON_THIS_MACHINE = START.replace("0.0.0.0", "127.0.0.1")


def start(folder, command, environment):
    """Run a start command in a folder, as a host would, with these environment variables added.

    A variable given as None is left out. exec makes the shell become the command, so that stopping
    the process stops the app.
    """
    variables = {name: value for name, value in {**os.environ, **environment}.items() if value is not None}
    return subprocess.Popen(f"exec {command}", shell=True, cwd=folder, env=variables,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)


def wait_until_up(address, path="/health", seconds=30):
    """The app's first answer at this path, or TimeoutError if nothing answers in that many seconds."""
    stop_at = time.monotonic() + seconds
    while time.monotonic() < stop_at:
        try:
            return requests.get(f"{address}{path}", timeout=2)
        except requests.ConnectionError:
            time.sleep(0.2)
    raise TimeoutError(f"nothing answered at {address}{path} within {seconds} seconds")


def stop(process):
    """Stop a started app, and wait until it has."""
    process.terminate()
    process.communicate(timeout=10)


key = secrets.token_urlsafe(24)                        # a key for this run of the notebook, never printed
LOCAL = "http://127.0.0.1:8040"
server = start(PROJECT, ON_THIS_MACHINE, {"PORT": "8040", "STATION_API_KEY": key})

print(ON_THIS_MACHINE)
print(wait_until_up(LOCAL).json())
print(requests.get(f"{LOCAL}/stations/tromso", timeout=10).json())


uvicorn main:app --host 127.0.0.1 --port $PORT
{'status': 'ok'}
{'id': 'tromso', 'name': 'Tromso', 'latitude': 69.65, 'longitude': 18.96}


The app runs as a process of its own now, as it will on the host, and it answered at port 8040
because `PORT` said so. `subprocess.Popen` starts a command and returns at once, while the command
goes on running, and `shell=True` runs it in a shell, which is what replaces `$PORT`. Until uvicorn
is listening, a request raises `ConnectionError`, which `wait_until_up` waits out. The key came from
`secrets.token_urlsafe`, which Python's documentation describes as a random URL-safe text string,
and it was made for this run.

### A key the app reads from the environment

Creating a plan needs the key, in an `X-API-Key` header. The key reached the app through the
environment of its process, as it will on the host, where it is set in the service's settings, and
it appears in none of the app's files:


In [5]:
plan = {"name": "Alta", "latitude": 69.97, "longitude": 23.27}
for label, headers in [("no key", {}), ("a wrong key", {"X-API-Key": "not-the-key"}), ("the key", {"X-API-Key": key})]:
    response = requests.post(f"{LOCAL}/plans", json=plan, headers=headers, timeout=10)
    print(f"{label:<12}", response.status_code, response.json())

written_in = sorted(path.name for path in PROJECT.iterdir() if path.is_file() and key in path.read_text())
print("files the key is written in:", written_in)


no key       401 {'detail': 'send a valid API key in X-API-Key'}
a wrong key  401 {'detail': 'send a valid API key in X-API-Key'}
the key      201 {'id': 1, 'name': 'Alta', 'latitude': 69.97, 'longitude': 23.27}
files the key is written in: []


The folder passes the Twelve-Factor App's test: it could be made public without giving the key away,
because it names the variable, and only the running process holds the value. Creating a plan without
the key, or with the wrong one, got `401 Unauthorized`, which the **Authentication** notebook said
means the server does not know who is asking.

### Checks before the files leave

`check` sends the requests a client will make first, to any address: the health route, the stations
compared with the practice API's, a plan refused without the key and created with it, and the
documentation page. It prints each result, and returns whether every check passed. Writing it for any
address lets the same checks run later against the hosted app:


In [6]:
def check(address, key, timeout=10):
    """Send the requests a client makes first, print each result, and say whether every one passed."""
    results = []

    def expect(label, passed, found):
        results.append(passed)
        print(f"  {'ok  ' if passed else 'FAIL'} {label}" + ("" if passed else f": {found}"))

    health = requests.get(f"{address}/health", timeout=timeout)
    expect("/health answers 200", health.status_code == 200, health.status_code)
    stations = requests.get(f"{address}/stations", timeout=timeout).json()
    expect("/stations matches the practice API's", stations == requests.get(f"{BASE}/stations", timeout=10).json(), stations)
    refused = requests.post(f"{address}/plans", json=plan, timeout=timeout)
    expect("a plan without the key gets 401", refused.status_code == 401, refused.status_code)
    created = requests.post(f"{address}/plans", json=plan, headers={"X-API-Key": key}, timeout=timeout)
    has_location = created.status_code == 201 and "Location" in created.headers
    expect("a plan with the key gets 201 and a Location", has_location, created.status_code)
    docs = requests.get(f"{address}/docs", timeout=timeout)
    expect("/docs answers 200", docs.status_code == 200, docs.status_code)
    return all(results)


print("every check passed:", check(LOCAL, key))


  ok   /health answers 200
  ok   /stations matches the practice API's
  ok   a plan without the key gets 401
  ok   a plan with the key gets 201 and a Location
  ok   /docs answers 200
every check passed: True


Every check passed against the app as the host will run it. A check that fails prints `FAIL`, what it
expected, and what it found, and the final line would say `False`.

### The files, packed for upload

The upload is the two files the host needs, and nothing else from the folder: running the app left a
`__pycache__` folder beside `main.py`, which a host makes for itself. `zipfile` packs the two files
by name. In Colab, `files.download` from `google.colab` sends the zip to your browser, and on your
own computer the zip is already on your disk:


In [7]:
stop(server)

archive = Path("scratch/station-api.zip")
with zipfile.ZipFile(archive, "w") as packed:
    for name in ["main.py", "requirements.txt"]:
        packed.write(PROJECT / name, arcname=name)

with zipfile.ZipFile(archive) as packed:
    print([(item.filename, item.file_size) for item in packed.infolist()])
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(archive))
else:
    print("The zip is at", archive)


[('main.py', 2194), ('requirements.txt', 33)]
The zip is at scratch/station-api.zip


### On Render

The deploy happens on two websites, so this section has no code. Render deploys from a Git
repository, and GitHub's website can hold one without Git on your computer.

1. **A repository.** On GitHub, choose the plus icon at the top right of any page, then **New
   repository**. Give it a name, such as `station-api`, choose whether it is public or private, and
   choose **Create repository**.
2. **The files.** Unzip `station-api.zip`. In the repository, choose **Add file**, then **Upload
   files**, drag `main.py` and `requirements.txt` into the page, type a commit message, and commit
   the files to the main branch.
3. **A web service.** On Render, sign in or sign up, choose **+ New** at the top right, then **Web
   Service**, connect your GitHub account when Render asks, and choose the repository.
4. **The settings.** Language `Python 3`, Build Command `pip install -r requirements.txt`, and Start
   Command `uvicorn main:app --host 0.0.0.0 --port $PORT`. Choose the **Free** plan.
5. **The secret.** Under **Environment Variables**, add `STATION_API_KEY`, with a key you make for
   this service, for instance with `secrets.token_urlsafe(24)` in a cell of your own. Keep a copy in
   Colab's Secrets panel under the same name, and print it nowhere.
6. **The deploy.** Create the service. Render's log shows the build installing the requirements and
   uvicorn starting, and once the service is live, its page shows an address ending in
   `onrender.com`.

Render gives that address HTTPS, and redirects plain HTTP to it. A free service spins down after 15
minutes without requests, and the next request waits about a minute while it starts again. Its plans
last only as long as the process, since they are kept in memory, and Render's documentation warns
that it may restart a free service at any time.

### The same checks, on this machine and at the hosted address

The pieces of this notebook, in one cell. The app starts again with the host's start command, and
`check` runs against it. Then, once the service is live and its address is pasted into `HOSTED`, the
same checks run against the hosted app, with the key read from Colab's Secrets panel, or from the
environment elsewhere, and a timeout long enough for a free service to wake. Until then, the cell
says what is missing:


In [8]:
HOSTED = ""                     # once the service is live, paste its address here, as https://station-api-xxxx.onrender.com

server = start(PROJECT, ON_THIS_MACHINE, {"PORT": "8040", "STATION_API_KEY": key})
wait_until_up(LOCAL)
print("on this machine, with the host's start command:")
print("  every check passed:", check(LOCAL, key))
stop(server)

print("at the hosted address:")
if HOSTED:
    if "google.colab" in sys.modules:
        from google.colab import userdata
        hosted_key = userdata.get("STATION_API_KEY")
    else:
        hosted_key = os.environ["STATION_API_KEY"]
    print("  every check passed:", check(HOSTED.rstrip("/"), hosted_key, timeout=90))
else:
    print("  no address yet: deploy the app on Render, paste its address into HOSTED, and run this cell again")


on this machine, with the host's start command:
  ok   /health answers 200
  ok   /stations matches the practice API's
  ok   a plan without the key gets 401
  ok   a plan with the key gets 201 and a Location
  ok   /docs answers 200
  every check passed: True
at the hosted address:
  no address yet: deploy the app on Render, paste its address into HOSTED, and run this cell again


### Where each part came from

| In the release | What it relies on | The section that showed it |
|---|---|---|
| `main.py`, with its own stations | a host that runs files, not cells | An app in a file of its own |
| `fastapi` and `uvicorn`, pinned | the versions this notebook ran | Requirements, pinned to what ran |
| `START`, with `$PORT` | the port the host chooses | The host's start command, on this machine |
| `STATION_API_KEY` in the environment | a secret kept out of every file | A key the app reads from the environment |
| `check(address, key)` | the same checks at any address | Checks before the files leave |
| `timeout=90` at the hosted address | a free service that takes about a minute to wake | On Render |
| `userdata.get("STATION_API_KEY")` | a key kept in Colab's Secrets panel | the **Authentication** notebook |

Every check passed on this machine, and the hosted address runs the same checks as soon as it is
pasted in. The code is the same in both places, so a check that passes here and fails there points
at what differs between the two machines, such as a variable missing from the service's settings or a
service still waking, rather than at the app.


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/18-hosting-an-api-solutions.ipynb).

**1.** Make a folder, `scratch/hello-api`, and write into it a `main.py` whose app answers `GET /`
with `{"hello": "stations"}`. Print the names of the files in the folder.


In [9]:
# your code here


**2.** Write the folder's `requirements.txt`, pinned to the versions this notebook runs, and print
it.


In [10]:
# your code here


**3.** Start the app with the host's start command, on `127.0.0.1` and port 8041, wait until `/`
answers, print the answer, and stop the app.


In [11]:
# your code here


**4.** Make the app answer with the value of an environment variable, `GREETING`, in place of
`stations`, and with `stations` when the variable is not set. Start it with `GREETING` set to
`Tromso`, print the answer, and stop it.


In [12]:
# your code here


**5.** Start the app again, print the status codes of `/docs` and `/openapi.json` and the title in
the OpenAPI document, and stop it.


In [13]:
# your code here


**6.** Pack the folder's `main.py` and `requirements.txt` into `scratch/hello-api.zip`, and print the
names inside the zip.


In [14]:
# your code here


## Common errors

### ERROR:    Error loading ASGI app. Could not import module "app".


In [15]:
failed = start(PROJECT, "uvicorn app:app --host 127.0.0.1 --port $PORT", {"PORT": "8040", "STATION_API_KEY": key})
print(failed.communicate(timeout=30)[0].strip().splitlines()[-1])


ERROR:    Error loading ASGI app. Could not import module "app".


`app:app` asks for a file named `app.py`, and the folder holds `main.py`: the name before the colon
is the module, and the name after it is the object in that module. uvicorn stopped at once, and on
Render this line would end the deploy's log, with a service that never goes live. Name the file the
app is in:


In [16]:
server = start(PROJECT, ON_THIS_MACHINE, {"PORT": "8040", "STATION_API_KEY": key})
print(wait_until_up(LOCAL).json())
stop(server)


{'status': 'ok'}


### KeyError: 'STATION_API_KEY'


In [17]:
failed = start(PROJECT, ON_THIS_MACHINE, {"PORT": "8040", "STATION_API_KEY": None})       # the variable left out
print(failed.communicate(timeout=30)[0].strip().splitlines()[-1])


KeyError: 'STATION_API_KEY'


`main.py` reads the key as it starts, the environment had no `STATION_API_KEY`, and `os.environ`
raised `KeyError`, so the app never started. On a host, this is the variable missing from the
service's settings, and the deploy's log ends with the same line. Failing as it starts is the better
failure: an app that started without its key would fail later, at the first request that needed it.
Set the variable:


In [18]:
server = start(PROJECT, ON_THIS_MACHINE, {"PORT": "8040", "STATION_API_KEY": key})
print(wait_until_up(LOCAL).json())
stop(server)


{'status': 'ok'}


### Error: Option '--port' requires an argument.


In [19]:
failed = start(PROJECT, ON_THIS_MACHINE, {"PORT": None, "STATION_API_KEY": key})          # no PORT, as on your own computer
print(failed.communicate(timeout=30)[0].strip().splitlines()[-1])


Error: Option '--port' requires an argument.


A host sets `PORT`, and your own computer usually does not, so the shell replaced `$PORT` with
nothing, and uvicorn got `--port` without a number. Set `PORT` for a run on your own computer, as
`start` did above, or give the command a port of its own there:


In [20]:
server = start(PROJECT, ON_THIS_MACHINE.replace("$PORT", "8040"), {"PORT": None, "STATION_API_KEY": key})
print(wait_until_up(LOCAL).json())
stop(server)


{'status': 'ok'}


### No error, and the key in the repository: a key written into a file


In [21]:
leaky = PROJECT / "settings.py"
leaky.write_text(f'API_KEY = "{key}"\n')                   # for main.py to import

written_in = sorted(path.name for path in PROJECT.iterdir() if path.is_file() and key in path.read_text())
print("files the key is written in:", written_in)


files the key is written in: ['settings.py']


Nothing failed, and the key is now in a file that the upload would put into the repository, where
anyone who can read the repository can read the key, and where it stays in the history even after
the file is deleted. A key that has been in a repository has to be treated as known, and replaced.
Keep the key in the environment, as `main.py` does, and delete the file:


In [22]:
leaky.unlink()

written_in = sorted(path.name for path in PROJECT.iterdir() if path.is_file() and key in path.read_text())
print("files the key is written in:", written_in)


files the key is written in: []


### ERROR:    Error loading ASGI app. Attribute "api" not found in module "main".


In [23]:
failed = start(PROJECT, ON_THIS_MACHINE.replace("main:app", "main:api"), {"PORT": "8040", "STATION_API_KEY": key})
print(failed.communicate(timeout=30)[0].strip().splitlines()[-1])


ERROR:    Error loading ASGI app. Attribute "api" not found in module "main".


uvicorn found `main.py`, and the object in it is called `app`, not `api`. The name after the colon
has to be the name of the `FastAPI()` object in the file, which a rename in the code, and not in the
start command, breaks. Name the object the file defines:


In [24]:
server = start(PROJECT, ON_THIS_MACHINE, {"PORT": "8040", "STATION_API_KEY": key})
print(wait_until_up(LOCAL).json())
stop(server)


{'status': 'ok'}


Last, the notebook is finished with its files, so this cell removes the scratch folder, with the app,
its zip, and the files the tasks wrote:


In [25]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A host runs files: the app in `main.py`, and a `requirements.txt` pinned to the versions you
  tested.
- The start command `uvicorn main:app --host 0.0.0.0 --port $PORT` listens on every address of the
  host's machine, at the port the host puts in `PORT`.
- A secret goes in the host's environment variables and is read with `os.environ`, so that no file
  holds it, and an app that needs it should fail as it starts when it is missing.
- Run the start command on your own machine, with `127.0.0.1` and a `PORT` of your own, and check the
  app there before any host sees it.
- Render deploys from a GitHub repository, gives the service an `onrender.com` address with HTTPS,
  and on its free plan spins the service down after 15 minutes without requests.
- The same checks, pointed at the hosted address, show whether the app works where its clients call
  it.


## What is next

That is the end of this guide. You can send a request and read what comes back, from its status code
and headers to JSON checked against a schema, send a credential, page through results, keep to a
rate limit, retry what is worth retrying, send data safely, wrap an API in a client with tests, and
build, validate, complete and host an API of your own.

The **Testing and Packaging** guide comes next. The tests in the **A Real Client** notebook and the
checks in this one were functions you wrote and ran yourself. That guide runs tests with pytest, lays
out a project so its code imports from anywhere, records an environment somebody else can reproduce,
and runs the tests on every push.

The guides after it move on to the libraries, and much of the data they start from arrives the way
this guide's did, as JSON from an API.


---

&#8592; **Previous:** [A Complete API](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/apis-and-json/17-a-complete-api.ipynb)  &nbsp;·&nbsp;  [APIs and JSON Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/apis-and-json.html)
